In [1]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Step 1: Imports + Path Setup
# =============================
import os, json, datetime, shutil, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import pandas as pd


print(tf.__version__)
SEED = 42


ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

2.19.0


In [3]:

# ---------------- PATH SETUP ----------------
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT70_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # with augmentation
TEST_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/Emotion_VGG16_70_30_DA_Finetune_{ts}"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- COPY DATA ----------------
def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

print("Copying Train/Val/Test to Colab local...")
fresh_copy(os.path.join(SPLIT70_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT70_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copy complete.")


Copying Train/Val/Test to Colab local...
✅ Copy complete.


In [4]:
# =============================
# Step 2: Data Generators (WITH augmentation for train)
# =============================
IMG_SIZE = (224, 224)
BATCH = 32


train_datagen = ImageDataGenerator(
preprocessing_function=preprocess_input,
rotation_range=15,
width_shift_range=0.1,
height_shift_range=0.1,
zoom_range=0.15,
shear_range=0.1,
horizontal_flip=True,
brightness_range=(0.9, 1.1),
fill_mode='nearest'
)


val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


flow_kwargs = dict(
target_size=IMG_SIZE,
batch_size=BATCH,
class_mode='categorical',
seed=SEED
)


train_gen = train_datagen.flow_from_directory(LOCAL_TRAIN, shuffle=True, **flow_kwargs)
val_gen = val_test_datagen.flow_from_directory(LOCAL_VAL, shuffle=False, **flow_kwargs)
test_gen = val_test_datagen.flow_from_directory(LOCAL_TEST, shuffle=False, **flow_kwargs)


class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

Found 28000 images belonging to 7 classes.
Found 8616 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [5]:
# Compute class weights to reduce imbalance impact
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for cls_name, idx in class_indices.items():
    dir_path = Path(LOCAL_TRAIN)/cls_name
    if dir_path.exists():
        counts[idx] = sum(1 for _ in dir_path.glob("**/*") if _.is_file())
class_weights = {i: float(np.max(counts) / c) if c > 0 else 1.0 for i, c in enumerate(counts)}
print("Class counts:", counts)
print("Class weights:", class_weights)

Class counts: [4000 4000 4000 4000 4000 4000 4000]
Class weights: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0}


In [6]:
# =============================
# Step 3: Model Builders
# =============================
LABEL_SMOOTH = 0.05


def build_head(x, units=256, dropout=0.5, l2=1e-4):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return out


def build_vgg16_model(unfreeze_from=None, units=256, dropout=0.5, l2=1e-4):
    base = VGG16(include_top=False, weights='imagenet', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    # Freeze all first; we'll selectively unfreeze later
    base.trainable = True
    if unfreeze_from is None:
        for layer in base.layers:
            layer.trainable = False
    else:
        set_trainable = False
        for layer in base.layers:
            if (unfreeze_from in layer.name):
                set_trainable = True
            layer.trainable = set_trainable
    # Keep BatchNorm (if any) in inference mode — VGG16 has none, but safe:
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    inp = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = base(inp, training=False)
    out = build_head(x, units=units, dropout=dropout, l2=l2)
    model = models.Model(inp, out)
    return model, base

In [8]:
# =============================
# Step 4: Lightweight Hyperparameter Tuning (warmup stage)
# =============================
# We quickly try a few combos to pick a good head size/dropout/L2 & LR for warmup.
param_grid = [
    {"units": 256, "dropout": 0.5, "l2": 1e-4, "lr": 1e-3},
    {"units": 512, "dropout": 0.4, "l2": 5e-5, "lr": 5e-4},
]

TUNE_EPOCHS = 2  # faster tuning
best_cfg, best_val = None, float('inf')
# Use only a subset of steps to speed up tuning
TUNE_FRAC = 0.35  # 35% of an epoch
train_steps = max(50, int(TUNE_FRAC * math.ceil(train_gen.samples / BATCH)))
val_steps   = max(30, int(TUNE_FRAC * math.ceil(val_gen.samples   / BATCH)))
print(f"[tune] using {train_steps} train steps & {val_steps} val steps per trial")

for cfg in param_grid:
    tf.keras.backend.clear_session()
    model, base = build_vgg16_model(unfreeze_from=None, units=cfg["units"], dropout=cfg["dropout"], l2=cfg["l2"])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg["lr"]),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
        metrics=['accuracy']
    )
    h = model.fit(
        train_gen,
        steps_per_epoch=train_steps,
        validation_data=val_gen,
        validation_steps=val_steps,
        epochs=TUNE_EPOCHS,
        verbose=1,
        class_weight=class_weights,
        # workers=2, # Removed unsupported argument
        # use_multiprocessing=True, # Removed unsupported argument
        # max_queue_size=16 # Removed unsupported argument
    )
    val_loss = h.history['val_loss'][-1]
    print("TUNE cfg:", cfg, "-> val_loss=", val_loss)
    if val_loss < best_val:
        best_val = val_loss
        best_cfg = cfg

print("Best warmup cfg:", best_cfg)
with open(Path(OUT_DIR)/'best_warmup_config.json', 'w') as f:
    json.dump(best_cfg, f, indent=2)

[tune] using 306 train steps & 94 val steps per trial


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/2
306/306 ━━━━━━━━━━━━━━━━━━━━ 179s 534ms/step - accuracy: 0.2298 - loss: 2.3760 - val_accuracy: 0.1712 - val_loss: 2.0632
Epoch 2/2
306/306 ━━━━━━━━━━━━━━━━━━━━ 201s 658ms/step - accuracy: 0.3193 - loss: 1.8555 - val_accuracy: 0.2148 - val_loss: 1.9910
TUNE cfg: {'units': 256, 'dropout': 0.5, 'l2': 0.0001, 'lr': 0.001} -> val_loss= 1.9909920692443848
Epoch 1/2
306/306 ━━━━━━━━━━━━━━━━━━━━ 168s 534ms/step - accuracy: 0.2551 - loss: 2.3281 - val_accuracy: 0.2257 - val_loss: 2.1213
Epoch 2/2
306/306 ━━━━━━━━━━━━━━━━━━━━ 163s 533ms/step - accuracy: 0.3191 - loss: 1.9371 - val_accuracy: 0.2676 - val_loss: 1.8761
TUNE cfg: {'units': 512, 'dropout': 0.4, 'l2': 5e-05, 'lr': 0.0005} -> val_loss= 1.8760939836502075
Best warmup cfg: {'units': 512, 'dropout': 0.4, 'l2': 5e-05, 'lr': 0.0005}


In [9]:
# Step 5: Full Training (Warmup → Finetune block5 → Finetune block4-5)
# =============================
# Phase A: Warmup (all frozen)
model, base = build_vgg16_model(unfreeze_from=None, units=best_cfg["units"], dropout=best_cfg["dropout"], l2=best_cfg["l2"])
model.compile(
optimizer=tf.keras.optimizers.Adam(learning_rate=best_cfg["lr"]),
loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
metrics=['accuracy']
)
ckpt_a = ModelCheckpoint(str(Path(OUT_DIR)/"best_vgg16_warmup.keras"), monitor='val_loss', save_best_only=True, verbose=1)
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1)


WARMUP_EPOCHS = 4
print("\n[ Phase A — Warmup (all frozen) ]\n")
hist_a = model.fit(train_gen, validation_data=val_gen, epochs=WARMUP_EPOCHS, callbacks=[ckpt_a, early, rlr], verbose=1, class_weight=class_weights)


# Phase B: Finetune last convolutional block (block5)
for layer in base.layers:
    layer.trainable = ('block5' in layer.name)


model.compile(
optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
metrics=['accuracy']
)
ckpt_b = ModelCheckpoint(str(Path(OUT_DIR)/"best_vgg16_finetune_block5.keras"), monitor='val_loss', save_best_only=True, verbose=1)


FINETUNE_B_EPOCHS = 8
print("\n[ Phase B — Finetune block5 ]\n")
hist_b = model.fit(train_gen, validation_data=val_gen, epochs=FINETUNE_B_EPOCHS, callbacks=[ckpt_b, early, rlr], verbose=1, class_weight=class_weights)


# Phase C: Finetune last two blocks (block4 + block5)
for layer in base.layers:
    layer.trainable = ('block4' in layer.name) or ('block5' in layer.name)


model.compile(
optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
metrics=['accuracy']
)
ckpt_c = ModelCheckpoint(str(Path(OUT_DIR)/"best_vgg16_finetune_block45.keras"), monitor='val_loss', save_best_only=True, verbose=1)


FINETUNE_C_EPOCHS = 7
print("\n[ Phase C — Finetune block4-5 ]\n")
hist_c = model.fit(train_gen, validation_data=val_gen, epochs=FINETUNE_C_EPOCHS, callbacks=[ckpt_c, early, rlr], verbose=1, class_weight=class_weights)


[ Phase A — Warmup (all frozen) ]

Epoch 1/4
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.2749 - loss: 2.1709
Epoch 1: val_loss improved from inf to 1.84382, saving model to /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_70_30_DA_Finetune_20250920_122754/best_vgg16_warmup.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 510s 579ms/step - accuracy: 0.2749 - loss: 2.1707 - val_accuracy: 0.3003 - val_loss: 1.8438 - learning_rate: 5.0000e-04
Epoch 2/4
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - accuracy: 0.3533 - loss: 1.7549
Epoch 2: val_loss improved from 1.84382 to 1.69653, saving model to /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_70_30_DA_Finetune_20250920_122754/best_vgg16_warmup.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 463s 530ms/step - accuracy: 0.3533 - loss: 1.7549 - val_accuracy: 0.3715 - val_loss: 1.6965 - learning_rate: 5.0000e-04
Epoch 3/4
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.3767 - loss: 1.6829
Epoch 3: val_loss improved from 1.69653 to 1.68693,

In [11]:
# =============================
# Step 6: Save histories & Curves
# =============================
for name, hist in [('warmup', hist_a), ('finetune_b5', hist_b), ('finetune_b45', hist_c)]:
    pd.DataFrame(hist.history).to_csv(Path(OUT_DIR)/f'history_{name}.csv', index=False)


acc = list(hist_a.history.get('accuracy', [])) + list(hist_b.history.get('accuracy', [])) + list(hist_c.history.get('accuracy', []))
val_acc = list(hist_a.history.get('val_accuracy', [])) + list(hist_b.history.get('val_accuracy', [])) + list(hist_c.history.get('val_accuracy', []))
loss = list(hist_a.history.get('loss', [])) + list(hist_b.history.get('loss', [])) + list(hist_c.history.get('loss', []))
val_loss = list(hist_a.history.get('val_loss', [])) + list(hist_b.history.get('val_loss', [])) + list(hist_c.history.get('val_loss', []))


epochs_range = range(1, len(acc)+1)
plt.figure(figsize=(8,6))
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy over Epochs'); plt.legend(); plt.tight_layout()
plt.savefig(Path(OUT_DIR)/'acc_curve.png', dpi=150)
plt.close()


plt.figure(figsize=(8,6))
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss over Epochs'); plt.legend(); plt.tight_layout()
plt.savefig(Path(OUT_DIR)/'loss_curve.png', dpi=150)
plt.close()

In [12]:
# =============================
# Step 7: Evaluate best checkpoint on TEST
# =============================
# Prefer last phase's best weights, but fallback if missing
best_ckpt = None
for fname in ["best_vgg16_finetune_block45.keras", "best_vgg16_finetune_block5.keras", "best_vgg16_warmup.keras"]:
    p = Path(OUT_DIR)/fname
    if p.exists():
        best_ckpt = p
        break

print("Using checkpoint:", best_ckpt)
best_model = tf.keras.models.load_model(best_ckpt)

# Evaluate
test_gen.reset()
test_loss, test_acc = best_model.evaluate(test_gen, verbose=1)
with open(Path(OUT_DIR)/'test_metrics.json', 'w') as f:
    json.dump({'test_loss': float(test_loss), 'test_accuracy': float(test_acc), 'classes': class_names}, f, indent=2)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")

# Predictions for reports
probs = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(probs, axis=1)
y_true = test_gen.classes


Using checkpoint: /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_70_30_DA_Finetune_20250920_122754/best_vgg16_finetune_block45.keras


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


225/225 ━━━━━━━━━━━━━━━━━━━━ 47s 202ms/step - accuracy: 0.5969 - loss: 1.3108
TEST -> loss: 1.1655 | acc: 0.6471
225/225 ━━━━━━━━━━━━━━━━━━━━ 41s 180ms/step


In [13]:
# Step 8: Reports — Classification report, Confusion matrix, ROC
# =============================
report_txt = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_txt)
(Path(OUT_DIR)/'classification_report.txt').write_text(report_txt)

rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
pd.DataFrame(rep_dict).transpose().to_csv(Path(OUT_DIR)/'classification_report.csv')

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(Path(OUT_DIR)/'confusion_matrix.csv')

plt.figure(figsize=(7,6))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion Matrix'); plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=45, ha='right')
plt.yticks(range(NUM_CLASSES), class_names)
th = cm.max()/2.
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > th else 'black')
plt.ylabel('True label'); plt.xlabel('Predicted label'); plt.tight_layout()
plt.savefig(Path(OUT_DIR)/'confusion_matrix.png', dpi=150)
plt.close()

# ROC (OvR)
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
roc_auc = {}
plt.figure(figsize=(8,6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
    roc_auc[i] = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")
# micro-average
fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), probs.ravel())
auc_micro = auc(fpr_micro, tpr_micro)
plt.plot([0,1],[0,1], linestyle=':')
plt.plot(fpr_micro, tpr_micro, linestyle='--', label=f"micro (AUC={auc_micro:.3f})")
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC (OvR)')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(Path(OUT_DIR)/'roc_curves.png', dpi=150)
plt.close()

with open(Path(OUT_DIR)/'roc_curves.json', 'w') as f:
    json.dump({'per_class_auc': {class_names[i]: float(roc_auc[i]) for i in range(NUM_CLASSES)},
               'micro_auc': float(auc_micro)}, f, indent=2)

print("\n✅ Done. Saved into:", OUT_DIR)
print("- best_vgg16_warmup.keras, best_vgg16_finetune_block5.keras, best_vgg16_finetune_block45.keras")
print("- history_warmup.csv, history_finetune_b5.csv, history_finetune_b45.csv")
print("- acc_curve.png, loss_curve.png, test_metrics.json, classification_report.(txt/csv), confusion_matrix.(png/csv), roc_curves.(png/json)")


              precision    recall  f1-score   support

       angry     0.6093    0.5762    0.5923       958
   disgusted     0.7200    0.4865    0.5806       111
     fearful     0.5172    0.3955    0.4483      1024
       happy     0.8656    0.8281    0.8464      1774
     neutral     0.5892    0.6399    0.6135      1233
         sad     0.4854    0.5726    0.5254      1247
   surprised     0.7299    0.7966    0.7618       831

    accuracy                         0.6471      7178
   macro avg     0.6452    0.6136    0.6240      7178
weighted avg     0.6502    0.6471    0.6460      7178


✅ Done. Saved into: /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_70_30_DA_Finetune_20250920_122754
- best_vgg16_warmup.keras, best_vgg16_finetune_block5.keras, best_vgg16_finetune_block45.keras
- history_warmup.csv, history_finetune_b5.csv, history_finetune_b45.csv
- acc_curve.png, loss_curve.png, test_metrics.json, classification_report.(txt/csv), confusion_matrix.(png/csv), roc_curves.(png